In [ ]:
import numpy as np
from django.contrib.gis.gdal.prototypes.geom import ogr_equals

from activators import SigmoidActivator, TanhActivator, IdentityActivator

In [ ]:
class LstmLayer(object):
    def __init__(self, input_width,state_width,learning_rate):
        self.input_width = input_width
        self.state_width = state_width
        self.learning_rate = learning_rate
        #门的激活函数
        self.gate_activator=SigmoidActivator()
        # 输出的激活函数
        self.output_activator=TanhActivator()

        self.times=0

        self.c_list=self.init_state_vec()
        self.h_list=self.init_state_vec()
        self.f_list=self.init_state_vec()
        self.i_list=self.init_state_vec()
        self.o_list=self.init_state_vec()
        self.ct_list=self.init_state_vec()

        self.Wfh,self.Wfx,self.bf=self.init_weight_mat()
        self.Wih,self.Wix,self.bi=self.init_weight_mat()
        self.Woh,self.Wox,self.bo=self.init_weight_mat()
        self.Wch,self.Wcx,self.bc=self.init_weight_mat()




    def init_state_vec(self):
        state_vec_list= [np.zeros((self.state_width, 1))]
        return state_vec_list
    def init_weight_mat(self):
        Wh=np.random.uniform(-1e-4,1e-4,(self.state_width,self.state_width))
        Wx=np.random.uniform(-1e-4,1e-4,(self.state_width,self.input_width))
        b=np.zeros((self.state_width,1))
        return Wh,Wx,b
    def forward(self,x):
        '''
        1. f = sigmoid(W_f·h_prev + U_f·x + b_f)  # 遗忘门
        2. i = sigmoid(W_i·h_prev + U_i·x + b_i)  # 输入门
        3. o = sigmoid(W_o·h_prev + U_o·x + b_o)  # 输出门
        4. ct~ = tanh(W_c·h_prev + U_c·x + b_c)   # 候选记忆
        5. c = f * c_prev + i * ct~               # 更新单元状态
        6. h = o * tanh(c)                        # 计算隐藏状态
        '''
        self.times+=1
        fg=self.calc_gate(x,self.Wfx,self.Wfh,self.bf,self.gate_activator)
        self.f_list.append(fg)
        ig=self.calc_gate(x,self.Wix,self.Wih,self.bi,self.gate_activator)
        self.i_list.append(ig)
        og=self.calc_gate(x,self.Wox,self.Woh,self.bo,self.gate_activator)
        self.o_list.append(og)
        ct=self.calc_gate(x,self.Wcx,self.Wch,self.bc,self.output_activator)
        self.ct_list.append(ct)
        c=fg*self.ct_list[self.times-1]+ig*ct
        self.c_list.append(c)
        h=og * self.output_activator.forward(c)
        self.h_list.append(h)




    def calc_gate(self,x,Wx,Wh,b,activator):
        h=self.h_list[self.times-1]
        net=np.dot(Wh,h)+np.dot(Wx,x)+b
        gate=activator.forward(net)
        return gate
    def backward(self,x,delta_h,activator):
        self.calc_delta(delta_h,activator)
        self.calc_gradient(x)

    def update(self):
        self.Wfh-=self.learning_rate*self.Wfh_grad
        self.Wfx-=self.learning_rate*self.Wfx_grad
        self.bf-=self.learning_rate*self.bf_grad

        self.Wih-=self.learning_rate*self.Wih_grad
        self.Wix-=self.learning_rate*self.Wix_grad
        self.bi-=self.learning_rate*self.bi_grad

        self.Woh-=self.learning_rate*self.Woh_grad
        self.Wox-=self.learning_rate*self.Wox_grad
        self.bo-=self.learning_rate*self.bo_grad

        self.Wch-=self.learning_rate*self.Wch_grad
        self.Wcx-=self.learning_rate*self.Wcx_grad
        self.bc-=self.learning_rate*self.bc_grad

    def calc_delta(self,delta_h,activator):
        self.delta_h_list =self.init_delta()
        self.delta_o_list=self.init_delta()
        self.delta_i_list=self.init_delta()
        self.delta_f_list=self.init_delta()
        self.delta_ct_list=self.init_delta()
        # 最后一个时刻的误差项应为从上一层传下来的误差
        self.delta_h_list[-1]=delta_h

        for k in range(self.times,0,-1):
            self.calc_delta_k(k)

    def init_delta(self):
        delta_list=[]
        for i in range(self.times+1):
            delta_list.append(np.zeros((self.state_width, 1)))
        return delta_list
    def calc_delta_k(self,k):
        ig=self.i_list[k]
        og=self.o_list[k]
        fg=self.f_list[k]
        ct=self.ct_list[k]
        c=self.c_list[k]
        c_prev=self.c_list[k-1]
        tanh_c=self.output_activator.forward(c)
        delta_k=self.delta_h_list[k]


        delta_o=(delta_k * tanh_c * self.gate_activator.backward(og))

        delta_f=(delta_k * og * (1-tanh_c *tanh_c) * c_prev * self.gate_activator.backward(fg))

        delta_i=(delta_k * og * (1-tanh_c *tanh_c) * ct * self.gate_activator.backward(ig))

        delta_ct=(delta_k * og * (1-tanh_c *tanh_c) * ig *self.output_activator.backward(ct))

        delta_h_prev=(np.dot(delta_o.transpose(),self.Woh)+np.dot(delta_i.transpose(),self.Wih)+np.dot(delta_f.transpose(),self.Wfh)+np.dot(delta_ct.transpose(),self.Wch)).transpose()

        self.delta_h_list[k-1]=delta_h_prev
        self.delta_f_list[k]=delta_f
        self.delta_i_list[k]=delta_i
        self.delta_o_list[k]=delta_o
        self.delta_ct_list[k]=delta_ct

        def calc_gradient(self,x):









